In [29]:
!pip install xgboost

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print(" テストデータ前処理ノート")
print("="*70)

# ========================================
# テストデータ読み込み
# ========================================

print("\n--- ステップ2: テストデータの読み込み ---")

# テストデータ読み込み
test_df = pd.read_parquet('../data/processed_test/test_features.parquet')
print(f"✅ テストデータ読み込み完了（{len(test_df):,}件）")

# -------------------------------------------------------------------
# 世帯数のカンマ除去
# -------------------------------------------------------------------
if '世帯数(万世帯)' in test_df.columns:
    test_df['世帯数(万世帯)'] = test_df['世帯数(万世帯)'].astype(str).str.replace(',', '', regex=False)
    test_df['世帯数(万世帯)'] = pd.to_numeric(test_df['世帯数(万世帯)'], errors='coerce')
    print("✅ 世帯数のカンマ除去完了")

categorical_cols_to_convert = ['最寄駅：名称', '市区町村名', '都道府県名']

# -------------------------------------------------------------------
# 0. 住宅ローン金利の追加
# -------------------------------------------------------------------
print("\n--- 0. 住宅ローン金利の追加 ---")

if '10年国債利回り(%)' in test_df.columns:
    test_df['住宅ローン金利推定'] = test_df['10年国債利回り(%)'] + 1.0
    print("✅ 住宅ローン金利推定を作成")

# -------------------------------------------------------------------
# 1. 欠損値の処理（KNN Imputation）
# -------------------------------------------------------------------
print("\n--- 1. KNN Imputationで築年数・駅距離の欠損補完 ---")

# 訓練データで学習したimputation_scalerとimputation_imputerを読み込み
# （訓練ノートブックで保存されていることが前提）

import pickle

try:
    # 訓練データで学習したオブジェクトを読み込み
    with open('../models/imputation_scaler.pkl', 'rb') as f:
        imputation_scaler = pickle.load(f)
    
    with open('../models/imputation_imputer.pkl', 'rb') as f:
        imputation_imputer = pickle.load(f)
    
    print("✅ Imputation関連オブジェクト読み込み完了")
    
    # 補完対象の列
    target_cols = ['取引時点での築年数', '最寄駅：距離（分）']
    helper_cols = ['市区町村コード', '面積（㎡）', '都市計画_高価格帯', '人口密度_log']
    cols_to_impute = target_cols + helper_cols
    
    # テストデータに適用
    X_test_impute = test_df[cols_to_impute].copy()
    X_test_scaled = imputation_scaler.transform(X_test_impute)
    X_test_imputed_scaled = imputation_imputer.transform(X_test_scaled)
    X_test_imputed = imputation_scaler.inverse_transform(X_test_imputed_scaled)
    
    # 結果を書き戻し
    test_df['取引時点での築年数'] = X_test_imputed[:, 0]
    test_df['最寄駅：距離（分）'] = X_test_imputed[:, 1]
    
    print(f"✅ KNN Imputation完了")
    print(f"   築年数の平均: {test_df['取引時点での築年数'].mean():.2f}")
    print(f"   駅距離の平均: {test_df['最寄駅：距離（分）'].mean():.2f}")

except FileNotFoundError:
    print("⚠️ 警告: Imputation関連ファイルが見つかりません")
    print("   訓練データで中央値補完を使用します")
    
    # フォールバック: 中央値補完
    target_cols = ['取引時点での築年数', '最寄駅：距離（分）']
    for col in target_cols:
        if col in test_df.columns and test_df[col].isnull().any():
            # 訓練データの中央値を使う（訓練ノートブックから取得）
            # ここでは仮にmedian_valuesが保存されていると仮定
            test_df[col] = test_df[col].fillna(test_df[col].median())

# -------------------------------------------------------------------
# 2. カテゴリ型に変換
# -------------------------------------------------------------------
print("\n--- 2. カテゴリ型に変換 ---")

for col in categorical_cols_to_convert:
    if col in test_df.columns:
        test_df[col] = test_df[col].astype('category')

print("✅ カテゴリ型変換完了")

# -------------------------------------------------------------------
# 3. その他の欠損値処理
# -------------------------------------------------------------------
print("\n--- 3. その他の欠損値処理 ---")

test_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# median_valuesを訓練データから読み込み（保存されている場合）
try:
    import json
    with open('../models/median_values.json', 'r', encoding='utf-8') as f:
        median_values = json.load(f)
    
    print("✅ 訓練データの中央値を読み込み")
    
    # 訓練データの中央値で欠損値を補完
    for col, median_val in median_values.items():
        if col in test_df.columns and test_df[col].isnull().any():
            test_df[col] = test_df[col].fillna(median_val)
            print(f"  {col}: 欠損値 {test_df[col].isnull().sum()}件を補完")

except FileNotFoundError:
    print("⚠️ 警告: median_values.jsonが見つかりません")
    print("   テストデータの中央値で補完します")
    
    # フォールバック: テストデータ自身の中央値
    for col in test_df.columns:
        if test_df[col].isnull().any():
            if test_df[col].dtype in ['float64', 'int64', 'float32', 'int32']:
                median_val = test_df[col].median()
                test_df[col] = test_df[col].fillna(median_val)

print(f"✅ 欠損値補完完了")

# -------------------------------------------------------------------
# 4. 保存
# -------------------------------------------------------------------
print("\n--- 4. 前処理済みテストデータを保存 ---")

import os
os.makedirs('../data/processed_test', exist_ok=True)

test_df.to_parquet('../data/processed_test/test_features.parquet', index=False)
print(f"✅ 保存完了: ../data/processed_test/test_features.parquet")

print("\n" + "="*70)
print("✅ テストデータ前処理完了")
print("="*70)
print(f"処理件数: {len(test_df):,}件")
print(f"特徴量数: {len(test_df.columns)}個")
print("="*70)

 テストデータ前処理ノート

--- ステップ2: テストデータの読み込み ---
✅ テストデータ読み込み完了（19,466件）
✅ 世帯数のカンマ除去完了

--- 0. 住宅ローン金利の追加 ---
✅ 住宅ローン金利推定を作成

--- 1. KNN Imputationで築年数・駅距離の欠損補完 ---
✅ Imputation関連オブジェクト読み込み完了
✅ KNN Imputation完了
   築年数の平均: 22.30
   駅距離の平均: 9.00

--- 2. カテゴリ型に変換 ---
✅ カテゴリ型変換完了

--- 3. その他の欠損値処理 ---
✅ 訓練データの中央値を読み込み
  建築年_西暦: 欠損値 0件を補完
  築年数_2乗: 欠損値 0件を補完
  築年数_3乗: 欠損値 0件を補完
  築年数_log: 欠損値 0件を補完
  築年数×面積: 欠損値 0件を補完
  築年数×駅距離: 欠損値 0件を補完
  築年数×建ぺい率: 欠損値 0件を補完
  築年数×容積率: 欠損値 0件を補完
  築年数×人口密度: 欠損値 0件を補完
  築年数×面積×駅距離: 欠損値 0件を補完
  地区名_頻度: 欠損値 0件を補完
  地区名_頻度_log: 欠損値 0件を補完
✅ 欠損値補完完了

--- 4. 前処理済みテストデータを保存 ---
✅ 保存完了: ../data/processed_test/test_features.parquet

✅ テストデータ前処理完了
処理件数: 19,466件
特徴量数: 90個


In [8]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print(" 2020年予測（本番）")
print("="*70)

# ========================================
# ステップ1: モデル・設定の読み込み
# ========================================
print("\n--- ステップ1: モデル・設定の読み込み ---")

# 設定ファイル
with open('../models/config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

USE_CLUSTERING = config['use_clustering']
N_CLUSTERS = config['n_clusters']
LGBM_WEIGHT = config['lgbm_weight']
XGB_WEIGHT = config['xgb_weight']
TARGET_COL = config['target_col']
USE_STACKING = config['use_stacking']

print(f"✅ 設定読み込み完了")
print(f"  クラスタリング: {'あり (クラスタ数 ' + str(N_CLUSTERS) + ')' if USE_CLUSTERING else 'なし'}")
print(f"  ブレンド比率: LGBM {LGBM_WEIGHT:.3f} / XGB {XGB_WEIGHT:.3f}")
print(f"  Stacking使用: {'はい' if USE_STACKING else 'いいえ'}")

# 特徴量リスト
with open('../models/feature_cols_final.json', 'r', encoding='utf-8') as f:
    FEATURE_COLS_FINAL = json.load(f)

print(f"✅ 特徴量リスト読み込み完了（{len(FEATURE_COLS_FINAL)}個）")

# カテゴリ列リスト
with open('../models/categorical_cols.json', 'r', encoding='utf-8') as f:
    categorical_cols_to_convert = json.load(f)

# XGBoost/LightGBMモデル
bst_xgb = xgb.Booster()
bst_xgb.load_model('../models/xgb_model_final.json')
print("✅ XGBoostモデル読み込み完了")

model_lgbm = lgb.Booster(model_file='../models/lgbm_model_final.txt')
print("✅ LightGBMモデル読み込み完了")

# Stackingメタモデル（使用する場合）
if USE_STACKING:
    with open('../models/meta_model.pkl', 'rb') as f:
        meta_model = pickle.load(f)
    print("✅ Stackingメタモデル読み込み完了")

# クラスタリング関連
if USE_CLUSTERING:
    with open('../models/scaler_final.pkl', 'rb') as f:
        scaler_final = pickle.load(f)
    
    with open('../models/kmeans_final.pkl', 'rb') as f:
        kmeans_final = pickle.load(f)
    
    with open('../models/cluster_feats.json', 'r', encoding='utf-8') as f:
        CLUSTER_FEATS_BASE = json.load(f)
    
    print("✅ クラスタリング関連読み込み完了")

# ========================================
# ステップ2: テストデータの読み込み
# ========================================
print("\n--- ステップ2: テストデータの読み込み ---")

# 🔥 前処理済みテストデータを読み込み
test_df = pd.read_parquet('../data/processed_test/test_features.parquet')
print(f"✅ テストデータ読み込み完了（{len(test_df):,}件）")

# IDカラムを保存（大文字のID）
test_ids = test_df['ID'].copy()
print(f"✅ ID保存完了")

# ========================================
# ステップ3: クラスタリング（必要な場合）
# ========================================
if USE_CLUSTERING:
    print("\n--- ステップ3: クラスタリング ---")
    
    # 数値特徴量のみ抽出
    X_cluster_test = scaler_final.transform(
        test_df[CLUSTER_FEATS_BASE].select_dtypes(include=[np.number])
    )
    
    # クラスタ予測
    test_df['Cluster_ID'] = kmeans_final.predict(X_cluster_test)
    
    print(f"✅ クラスタリング完了")
    
    # クラスタごとの件数
    print("\n【クラスタごとの件数】")
    for cluster_id in range(N_CLUSTERS):
        count = (test_df['Cluster_ID'] == cluster_id).sum()
        print(f"  Cluster {cluster_id}: {count:,}件")

# ========================================
# ステップ4: 特徴量準備
# ========================================
print("\n--- ステップ4: 特徴量準備 ---")

# 特徴量の存在チェック
missing_features = [col for col in FEATURE_COLS_FINAL if col not in test_df.columns]
if missing_features:
    print(f"⚠️ 警告: 以下の特徴量がテストデータにありません:")
    for col in missing_features[:10]:
        print(f"  - {col}")
    if len(missing_features) > 10:
        print(f"  ... 計 {len(missing_features)}個")
    
    # 存在す

 2020年予測（本番）

--- ステップ1: モデル・設定の読み込み ---
✅ 設定読み込み完了
  クラスタリング: なし
  ブレンド比率: LGBM 0.855 / XGB 0.145
  Stacking使用: はい
✅ 特徴量リスト読み込み完了（88個）
✅ XGBoostモデル読み込み完了
✅ LightGBMモデル読み込み完了
✅ Stackingメタモデル読み込み完了

--- ステップ2: テストデータの読み込み ---
✅ テストデータ読み込み完了（19,466件）
✅ ID保存完了

--- ステップ4: 特徴量準備 ---
⚠️ 警告: 以下の特徴量がテストデータにありません:
  - 間取り_grouped_オープンフロア
  - 間取り_grouped_欠損値
  - 間取り_grouped_１ＤＫ
  - 間取り_grouped_１Ｒ
  - 間取り_grouped_２ＤＫ
  - 間取り_grouped_２Ｋ
  - 間取り_grouped_２ＬＤＫ＋Ｓ
  - 間取り_grouped_３ＤＫ
  - 間取り_grouped_４ＤＫ


In [10]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print(" 2020年予測（本番）")
print("="*70)

# ========================================
# ステップ1: モデル・設定の読み込み
# ========================================
print("\n--- ステップ1: モデル・設定の読み込み ---")

# 設定ファイル
with open('../models/config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

USE_CLUSTERING = config['use_clustering']
N_CLUSTERS = config['n_clusters']
LGBM_WEIGHT = config['lgbm_weight']
XGB_WEIGHT = config['xgb_weight']
TARGET_COL = config['target_col']
USE_STACKING = config['use_stacking']

print(f"✅ 設定読み込み完了")
print(f"  クラスタリング: {'あり (クラスタ数 ' + str(N_CLUSTERS) + ')' if USE_CLUSTERING else 'なし'}")
print(f"  ブレンド比率: LGBM {LGBM_WEIGHT:.3f} / XGB {XGB_WEIGHT:.3f}")
print(f"  Stacking使用: {'はい' if USE_STACKING else 'いいえ'}")

# 特徴量リスト
with open('../models/feature_cols_final.json', 'r', encoding='utf-8') as f:
    FEATURE_COLS_FINAL = json.load(f)

print(f"✅ 特徴量リスト読み込み完了（{len(FEATURE_COLS_FINAL)}個）")

# カテゴリ列リスト
with open('../models/categorical_cols.json', 'r', encoding='utf-8') as f:
    categorical_cols_to_convert = json.load(f)

# XGBoost/LightGBMモデル
bst_xgb = xgb.Booster()
bst_xgb.load_model('../models/xgb_model_final.json')
print("✅ XGBoostモデル読み込み完了")

model_lgbm = lgb.Booster(model_file='../models/lgbm_model_final.txt')
print("✅ LightGBMモデル読み込み完了")

# Stackingメタモデル（使用する場合）
if USE_STACKING:
    with open('../models/meta_model.pkl', 'rb') as f:
        meta_model = pickle.load(f)
    print("✅ Stackingメタモデル読み込み完了")

# クラスタリング関連
if USE_CLUSTERING:
    with open('../models/scaler_final.pkl', 'rb') as f:
        scaler_final = pickle.load(f)
    
    with open('../models/kmeans_final.pkl', 'rb') as f:
        kmeans_final = pickle.load(f)
    
    with open('../models/cluster_feats.json', 'r', encoding='utf-8') as f:
        CLUSTER_FEATS_BASE = json.load(f)
    
    print("✅ クラスタリング関連読み込み完了")

# ========================================
# ステップ2: テストデータの読み込み
# ========================================
print("\n--- ステップ2: テストデータの読み込み ---")

# 前処理済みテストデータを読み込み
test_df = pd.read_parquet('../data/processed_test/test_features.parquet')
print(f"✅ テストデータ読み込み完了（{len(test_df):,}件）")

# IDカラムを保存（大文字のID）
test_ids = test_df['ID'].copy()
print(f"✅ ID保存完了")

# ========================================
# ステップ3: クラスタリング（必要な場合）
# ========================================
if USE_CLUSTERING:
    print("\n--- ステップ3: クラスタリング ---")
    
    # 数値特徴量のみ抽出
    X_cluster_test = scaler_final.transform(
        test_df[CLUSTER_FEATS_BASE].select_dtypes(include=[np.number])
    )
    
    # クラスタ予測
    test_df['Cluster_ID'] = kmeans_final.predict(X_cluster_test)
    
    print(f"✅ クラスタリング完了")
    
    # クラスタごとの件数
    print("\n【クラスタごとの件数】")
    for cluster_id in range(N_CLUSTERS):
        count = (test_df['Cluster_ID'] == cluster_id).sum()
        print(f"  Cluster {cluster_id}: {count:,}件")

# ========================================
# ステップ4: 特徴量準備（ダミー変数の補完）
# ========================================
print("\n--- ステップ4: 特徴量準備 ---")

# 🔥 重要: モデルが期待する全ての特徴量を揃える
missing_features = [col for col in FEATURE_COLS_FINAL if col not in test_df.columns]

if missing_features:
    print(f"⚠️ 不足している特徴量を0で補完します:")
    for col in missing_features:
        test_df[col] = 0
        print(f"  + {col}")
    print(f"  計 {len(missing_features)}個を追加")
else:
    print(f"✅ すべての特徴量が存在します")

# 特徴量抽出（順序も揃える）
X_test = test_df[FEATURE_COLS_FINAL].copy()

# 数値列のみfloat32
numeric_cols = X_test.select_dtypes(include=[np.number]).columns
X_test[numeric_cols] = X_test[numeric_cols].astype(np.float32)

print(f"✅ 特徴量準備完了（{len(FEATURE_COLS_FINAL)}個）")

# ========================================
# ステップ5: 予測
# ========================================
print("\n--- ステップ5: 予測 ---")

# XGBoost予測
dtest_xgb = xgb.DMatrix(X_test, enable_categorical=True)
pred_xgb_log = bst_xgb.predict(dtest_xgb)
print(f"✅ XGBoost予測完了")
print(f"  予測範囲（log）: [{pred_xgb_log.min():.4f}, {pred_xgb_log.max():.4f}]")

# LightGBM予測
pred_lgbm_log = model_lgbm.predict(X_test)
print(f"✅ LightGBM予測完了")
print(f"  予測範囲（log）: [{pred_lgbm_log.min():.4f}, {pred_lgbm_log.max():.4f}]")

# ブレンド
pred_blended_log = (pred_lgbm_log * LGBM_WEIGHT) + (pred_xgb_log * XGB_WEIGHT)
print(f"✅ ブレンド完了")
print(f"  予測範囲（log）: [{pred_blended_log.min():.4f}, {pred_blended_log.max():.4f}]")

# Stacking（使用する場合）
if USE_STACKING:
    X_meta_test = np.column_stack([pred_lgbm_log, pred_xgb_log])
    pred_final_log = meta_model.predict(X_meta_test)
    print(f"✅ Stacking予測完了")
    print(f"  予測範囲（log）: [{pred_final_log.min():.4f}, {pred_final_log.max():.4f}]")
else:
    pred_final_log = pred_blended_log
    print(f"✅ 最終予測: ブレンドを使用")

# ========================================
# ステップ6: 提出ファイル作成（log空間のまま）
# ========================================
print("\n--- ステップ6: 提出ファイル作成 ---")

# log空間のまま提出ファイル作成
submission = pd.DataFrame({
    'ID': test_ids,
    TARGET_COL: pred_final_log
})

# CSV保存
import os
os.makedirs('../submission', exist_ok=True)

submission.to_csv('../submission/submission_final.csv', index=False)
print(f"✅ 提出ファイル保存完了")
print(f"  保存場所: ../submission/submission_final.csv")

# 統計情報表示
print("\n【提出ファイルの統計（log空間）】")
print(submission[TARGET_COL].describe())

# 参考：元のスケールでの統計
pred_original_scale = np.exp(pred_final_log)
print("\n【参考：元のスケール（万円）】")
print(f"  最小値: {pred_original_scale.min():.0f}万円")
print(f"  最大値: {pred_original_scale.max():.0f}万円")
print(f"  平均値: {pred_original_scale.mean():.0f}万円")
print(f"  中央値: {np.median(pred_original_scale):.0f}万円")

# 先頭5行を表示
print("\n【提出ファイルの先頭5行】")
print(submission.head())

print("\n" + "="*70)
print("🎊 予測完了！")
print("="*70)
print(f"予測件数: {len(submission):,}件")
print(f"提出ファイル: ../submission/submission_final.csv")
print(f"カラム: ID, {TARGET_COL}")
print("="*70)

 2020年予測（本番）

--- ステップ1: モデル・設定の読み込み ---
✅ 設定読み込み完了
  クラスタリング: なし
  ブレンド比率: LGBM 0.855 / XGB 0.145
  Stacking使用: はい
✅ 特徴量リスト読み込み完了（88個）
✅ XGBoostモデル読み込み完了
✅ LightGBMモデル読み込み完了
✅ Stackingメタモデル読み込み完了

--- ステップ2: テストデータの読み込み ---
✅ テストデータ読み込み完了（19,466件）
✅ ID保存完了

--- ステップ4: 特徴量準備 ---
⚠️ 不足している特徴量を0で補完します:
  + 間取り_grouped_オープンフロア
  + 間取り_grouped_欠損値
  + 間取り_grouped_１ＤＫ
  + 間取り_grouped_１Ｒ
  + 間取り_grouped_２ＤＫ
  + 間取り_grouped_２Ｋ
  + 間取り_grouped_２ＬＤＫ＋Ｓ
  + 間取り_grouped_３ＤＫ
  + 間取り_grouped_４ＤＫ
  計 9個を追加
✅ 特徴量準備完了（88個）

--- ステップ5: 予測 ---
✅ XGBoost予測完了
  予測範囲（log）: [5.9531, 8.4442]
✅ LightGBM予測完了
  予測範囲（log）: [5.9916, 8.6514]
✅ ブレンド完了
  予測範囲（log）: [6.0468, 8.6214]
✅ Stacking予測完了
  予測範囲（log）: [6.0610, 8.6075]

--- ステップ6: 提出ファイル作成 ---
✅ 提出ファイル保存完了
  保存場所: ../submission/submission_final.csv

【提出ファイルの統計（log空間）】
count    19466.000000
mean         7.273680
std          0.273898
min          6.061030
25%          7.124875
50%          7.304394
75%          7.459066
max          8.607517
Name: 取引価格（総額）_log,